<a href="https://colab.research.google.com/github/sherjahong1r/Machine-Learning-Lessons/blob/main/13_Deployment_qilish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Modelni moslashtirish**

## **Avval shug'ullantirilgan modelni o'z ma'lumotimizga moslashtirish**

In [ ]:
!pip install -U datasets fsspec
# Bu buyruq datasets va fsspec kutubxonalarini o'rnatadi yoki ularni eng so'nggi versiyaga yangilaydi.
# datasets ma'lumotlar to'plamlari bilan ishlash uchun, fsspec esa turli fayl tizimlari bilan bog'lanish uchun ishlatiladi.
# -U (yoki --upgrade) belgisi esa, agar bu kutubxonalar allaqachon o'rnatilgan bo'lsa, ularni eng so'nggi mavjud versiyasiga yangilashni bildiradi.

In [ ]:
!pip install -U transformers
# Bu  buyrug'i transformers kutubxonasini o'rnatadi yoki uni eng so'nggi versiyasiga yangilaydi. transformers - bu
# Hugging Face kompaniyasining kutubxonasi bo'lib, u tayyorlangan (pre-trained) modellar, masalan, BERT, GPT-2, T5 kabi
# modellarni yuklash, ulardan foydalanish va o'z ma'lumotlaringizga moslashtirish (finetuning) uchun ishlatiladi.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
small_train = dataset["train"].shuffle(seed=42).select([1 for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([1 for i in list(range(300))])

# Bu kod datasets kutubxonasidan foydalanib "imdb" ma'lumotlar to'plamini yuklaydi. So'ngra,
# u o'qitish uchun 1000 ta va testlash uchun 300 ta misoldan iborat kichik qismlarni tasodifiy tanlab oladi.
# Bu modelni tezroq sinovdan o'tkazish uchun mo'ljallangan.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

# Bu kod AutoTokenizer yordamida distilbert-base-uncased modeliga mos keladigan tokenizatorni yuklaydi.
# preprocess_function esa matnni tokenizatsiya qilish, qirqish (truncation) va to'ldirish (padding) orqali
# modelga kiritish uchun tayyorlaydi. Yakunda, bu funksiya small_train va small_test ma'lumotlar to'plamlariga
# qo'llanilib, ularni tokenizatsiyalangan formatga o'tkazadi.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Bu kod transformers kutubxonasidan AutoModelForSequenceClassification klassini import qiladi va distilbert-base-uncased
# modelini ikki sinf (num_labels=2) uchun moslashtirilgan tarzda yuklaydi. Bu odatda matn tasniflash vazifalari uchun
# tayyorlangan modelni ishga tushirishni anglatadi.
# num_labels=2 bu modelning qancha chiqish sinflariga ega bo'lishini bildiradi. Bizning holatimizda, IMDB ma'lumotlar to'plami
# bilan ishlayapmiz, bu erda har bir kinofilm sharhi ikki toifadan biriga bo'linadi: ijobiy (positive) yoki salbiy (negative).
# 0 (salbiy) - sharh salbiy fikrni bildiradi.
# 1 (ijobiy) - sharh ijobiy fikrni bildiradi.
# Shuning uchun, model sharhlarni ikkita guruhga ajratishi kerak, ya'ni ikkita "label" yoki "sinf" mavjud. Shu sababli,
# num_labels qiymati 2 ga teng qilib belgilangan. Agar siz ko'proq sinflarga ega bo'lgan boshqa turdagi tasniflash vazifasi
# bilan ishlayotgan bo'lsangiz, bu qiymat o'zgarishi mumkin edi (masalan, uchta sinf bo'lsa num_labels=3 bo'ladi).

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=10,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

# qisqacha qilib aytganda: TrainingArguments orqali modelni o'qitish parametrlarini
# (masalan, epoxalar soni, batch hajmi) sozlaydi, so'ngra Trainer obyektini yaratib, model, o'qitish parametrlari,
# o'qitish va baholash ma'lumotlarini unga bog'laydi. Eng oxirida esa trainer.train() buyrug'i bilan modelni o'qitish jarayonini boshlaydi.



In [ ]:
# results = trainer.evaluate()
# print(results)

In [ ]:
texts = [
    "This movie was absolutely lovely",
    "This movie was absolutely awful",
    "Just ok movie, not so bad"
]

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cuda')

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

label_map = {0: "negative", 1: 'possitive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())

# Bu kod matnlarni (texts) model tushunadigan shaklga (inputs) o'tkazadi, model (GPU yordamida) ularning
# sentimentini tahmin qiladi (outputs), bu tahminlarni ehtimollikka (probs) aylantiradi, eng yuqori ehtimolli
# sinfni (predictions) tanlaydi va nihoyat, har bir matn uchun tahminni (salbiy/ijobiy) va uning ishonchliligini chop etadi.

# **DEPLOYMENT**

# **Website yaratish va modeldan hayotda foydalanish**

In [ ]:
import torch

def predict_sentiment(text):
  inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cuda')

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1).item()

  return "Ijobiy" if predictions == 1 else "Salbiy"

  # Bu predict_sentiment funksiyasi berilgan matnning ('text') sentimentini (ijobiy yoki salbiy) aniqlash uchun
  # ishlatiladi. U matnni tokenizatsiya qiladi, modeldan o'tkazib tahmin ehtimolliklarini oladi, eng yuqori
  # ehtimolga ega sinfni tanlaydi va natijani 'Ijobiy' yoki 'Salbiy' ko'rinishida qaytaradi.

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=predict_sentiment,
    inputs="text",
    outputs="text",
    title="Sentiment tahlil qilish",
    description="Bu yerda sentiment analiz qilish mumkin"
)

interface.launch(share=True)

# Bu kod Gradio kutubxonasi yordamida kichik veb-interfeys yaratadi. predict_sentiment funksiyasini kirish sifatida
# 'text' olib, natijani 'text' ko'rinishida chiqaradi. title va description interfeysning sarlavhasi va tavsifini
# belgilaydi. interface.launch(share=True) esa bu veb-interfeyni ishga tushirib, uni umumiy foydalanish uchun havola yaratadi.

In [ ]:
torch.save(model.state_dict(), 'model.pth')
# Bu kod model.state_dict() yordamida modelning o'qitilgan parametrlarini (vazn va boshqa konfiguratsiyalarini)
# oladi va ularni 'model.pth' nomli faylga saqlaydi. Bu modelni keyinchalik qayta yuklash va ishlatish imkonini beradi.

In [ ]:
model.load_state_dict(torch.load('model.pth'))
model.eval()

# Bu kod model.load_state_dict(torch.load('model.pth')) yordamida 'model.pth' faylidan saqlangan model
# parametrlarini (vaznlarini) yuklaydi. model.eval() esa modelni baholash rejimiga o'tkazadi, bu inferensiya
# (tahmin qilish) paytida optimallashtirishlar (masalan, Dropout qatlamlarini o'chirib qo'yish) uchun muhimdir.